# ============================================================
# 01_treinamento_bert_light.ipynb
# ============================================================
# Aula prática: fine-tuning rápido do BERT (execução leve)
# ============================================================


In [1]:
# ------------------------------------------------------------
# 🔹 Etapa 1: Importação de bibliotecas e verificação da GPU
# ------------------------------------------------------------
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Nome da GPU:', torch.cuda.get_device_name(0))

GPU disponível: True
Nome da GPU: NVIDIA GeForce RTX 3060 Ti


In [2]:
# ------------------------------------------------------------
# 🔹 Etapa 2: Carregar e inspecionar o dataset
# ------------------------------------------------------------
# Dataset leve: apenas 500 amostras de treino e 200 de teste
dataset = load_dataset('imdb')

train_dataset = dataset['train'].shuffle(seed=42).select(range(500))
test_dataset = dataset['test'].shuffle(seed=42).select(range(200))

print(f'Tamanho treino: {len(train_dataset)} | Tamanho teste: {len(test_dataset)}')

Tamanho treino: 500 | Tamanho teste: 200


In [3]:
# ------------------------------------------------------------
# 🔹 Etapa 3: Tokenização
# ------------------------------------------------------------
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize, batched=True, batch_size=1000)
tokenized_test = test_dataset.map(tokenize, batched=True, batch_size=1000)

tokenized_train = tokenized_train.rename_column('label', 'labels')
tokenized_test = tokenized_test.rename_column('label', 'labels')

tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [4]:
# ------------------------------------------------------------
# 🔹 Etapa 4: Carregar modelo pré-treinado
# ------------------------------------------------------------
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
# ------------------------------------------------------------
# 🔹 Etapa 5: Definir métricas
# ------------------------------------------------------------
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

In [6]:
# ------------------------------------------------------------
# 🔹 Etapa 6: Argumentos de treinamento
# ------------------------------------------------------------
training_args = TrainingArguments(
    output_dir='./results_light',
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir='./logs_light',
    logging_steps=10,
    save_total_limit=1,
    push_to_hub=False,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [7]:
# ------------------------------------------------------------
# 🔹 Etapa 7: Inicializar Trainer e treinar
# ------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.656446,0.657148,0.615000,0.583784,0.606742,0.562500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=63, training_loss=0.6657574309243096, metrics={'train_runtime': 9.4537, 'train_samples_per_second': 52.889, 'train_steps_per_second': 6.664, 'total_flos': 32888881920000.0, 'train_loss': 0.6657574309243096, 'epoch': 1.0})

In [8]:
# ------------------------------------------------------------
# 🔹 Etapa 8: Avaliação final
# ------------------------------------------------------------
metrics = trainer.evaluate()
print('\nMétricas finais:', metrics)


Métricas finais: {'eval_loss': 0.6571478247642517, 'eval_accuracy': 0.615, 'eval_f1': 0.5837837837837838, 'eval_precision': 0.6067415730337079, 'eval_recall': 0.5625, 'eval_runtime': 0.6344, 'eval_samples_per_second': 315.237, 'eval_steps_per_second': 39.405, 'epoch': 1.0}


In [9]:
# ------------------------------------------------------------
# 🔹 Etapa 9: Salvar modelo e tokenizador
# ------------------------------------------------------------
OUTPUT_DIR = './bert-imdb-light-model' 

# Salva os pesos treinados e a configuração do modelo
trainer.save_model(OUTPUT_DIR) 

# Salva o vocabulário e a configuração do tokenizador
tokenizer.save_pretrained(OUTPUT_DIR) 

print(f'\n✅ Treinamento leve concluído! Modelo salvo em {OUTPUT_DIR}')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Treinamento leve concluído! Modelo salvo em ./bert-imdb-light-model


In [10]:
# ------------------------------------------------------------
# 🔹 Etapa 10: Carregar Modelo Salvo para Inferência
# ------------------------------------------------------------
from transformers import BertTokenizerFast, BertForSequenceClassification
import torch

MODEL_PATH = './bert-imdb-light-model' 

# Carregar o Tokenizador e o Modelo SALVOS
tokenizer_inf = BertTokenizerFast.from_pretrained(MODEL_PATH)
model_inf = BertForSequenceClassification.from_pretrained(MODEL_PATH)

# Definir dispositivo (GPU ou CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_inf.to(device)
model_inf.eval() # Coloca o modelo em modo de avaliação (importante para inferência)

print(f'Modelo carregado para inferência no dispositivo: {device}')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo carregado para inferência no dispositivo: cuda


In [11]:
# ------------------------------------------------------------
# 🔹 Etapa 11: Função de Previsão de Sentimento
# ------------------------------------------------------------

def prever_sentimento(texto):
    # Tokenizar o novo texto
    inputs = tokenizer_inf(texto, 
                           padding='max_length', 
                           truncation=True, 
                           max_length=128, 
                           return_tensors='pt')
    
    # Mover inputs para a GPU (se aplicável)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Rodar a inferência
    with torch.no_grad():
        outputs = model_inf(**inputs)
    
    # Obter probabilidades (Softmax)
    probabilities = torch.softmax(outputs.logits, dim=1)
    
    # Obter a previsão final (o índice com maior probabilidade)
    predicted_class_id = torch.argmax(probabilities, dim=1).item()
    
    # Mapeamento do rótulo (0=Negativo, 1=Positivo no IMDB)
    labels = {0: "NEGATIVO 👎", 1: "POSITIVO 👍"}
    sentimento = labels[predicted_class_id]
    
    # Retorna o sentimento e as probabilidades formatadas
    return sentimento, probabilities.cpu().numpy()[0]

print('Função de previsão definida.')

Função de previsão definida.


In [19]:
# ------------------------------------------------------------
# 🔹 Etapa 12: Inferência em Textos Reais
# ------------------------------------------------------------

print("--- Testes de Inferência em Novas Frases ---")

# Exemplo 1: Sentimento Positivo
texto_positivo = "This is the best film of this year. Amazing!"
sentimento, probs = prever_sentimento(texto_positivo)
print(f"Texto: '{texto_positivo}'")
print(f"Predição: **{sentimento}** (Probabilidade Positiva: {probs[1]:.4f})\n")

# Exemplo 2: Sentimento Negativo
texto_negativo = "The plot was amazing, the acting was graceful, and I wanted pay the ticket again immediately."
sentimento, probs = prever_sentimento(texto_negativo)
print(f"Texto: '{texto_negativo}'")
print(f"Predição: **{sentimento}** (Probabilidade Negativa: {probs[0]:.4f})\n")

# Exemplo 3: Sentimento Neutro/Ambíguo
texto_neutro = "The movie had a worst first half and completely fell apart in the final act."
sentimento, probs = prever_sentimento(texto_neutro)
print(f"Texto: '{texto_neutro}'")
print(f"Predição: **{sentimento}** (Probabilidade Positiva: {probs[1]:.4f}, Negativa: {probs[0]:.4f})\n")

--- Testes de Inferência em Novas Frases ---
Texto: 'This is the best film of this year. Amazing!'
Predição: **POSITIVO 👍** (Probabilidade Positiva: 0.5595)

Texto: 'The plot was amazing, the acting was graceful, and I wanted pay the ticket again immediately.'
Predição: **POSITIVO 👍** (Probabilidade Negativa: 0.4501)

Texto: 'The movie had a worst first half and completely fell apart in the final act.'
Predição: **NEGATIVO 👎** (Probabilidade Positiva: 0.4852, Negativa: 0.5148)

